# Notebook 04: Tier-2 End-to-End ResNet-18 & Dynamic Leaf Expansion

This notebook orchestrates the Feature F5 experiments on remote JupyterHub:
1. Dynamic entropy-based leaf sub-clustering ablation on CIFAR-100.
2. Tier-2 End-to-End ResNet-18 vision pipeline across negative sampling strategies.
3. Periodic Ward Confusion Tree refresh during end-to-end training.

In [1]:
import os
import sys
from pathlib import Path
import torch
import numpy as np
import matplotlib.pyplot as plt

# Ensure project root is in sys.path
REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.utils.experiment_config import (
    ExecutionMode,
    ExperimentConfig,
    get_isolated_paths,
    print_execution_banner,
)
from src.utils.seed import set_global_seed
from src.data.cifar100_images import load_cifar100_raw_images
from src.trees.semantic_tree import build_semantic_tree
from src.mcts.expansion import maybe_expand_leaf, expand_leaf_by_entropy, compute_prediction_entropy
from src.training.trainer import train_scenario1
from src.training.e2e_trainer import (
    Tier2ResNet18,
    train_e2e_scenario1,
    train_e2e_scenario2,
    E2EExperimentConfig,
    run_e2e_experiment,
)

# Execution mode: REAL by default for scientific runs; SMOKE for testing
EXECUTION_MODE = ExecutionMode.REAL
if os.environ.get("EXECUTION_MODE"):
    EXECUTION_MODE = ExecutionMode(os.environ["EXECUTION_MODE"].lower())
elif "pytest" in sys.modules:
    EXECUTION_MODE = ExecutionMode.SMOKE

config = ExperimentConfig(execution_mode=EXECUTION_MODE)
print_execution_banner(config, notebook_name="04_tier2_e2e_and_expansion")
paths = get_isolated_paths(config, repo_root=REPO_ROOT)

SEED = set_global_seed(42)
RESULTS_DIR = paths.results_dir
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINTS_DIR = paths.checkpoints_dir
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Repository root: {REPO_ROOT} | Results: {RESULTS_DIR} | Checkpoints: {CHECKPOINTS_DIR} | Device: {device}")


Using device: cuda


In [ ]:
# Load CIFAR-100 raw 32x32 images according to execution mode contract
train_images, train_labels = load_cifar100_raw_images(
    data_dir=REPO_ROOT / "data",
    split="train",
    download=(config.execution_mode == ExecutionMode.REAL),
    config=config,
    num_smoke_samples=100,
)

test_images, test_labels = load_cifar100_raw_images(
    data_dir=REPO_ROOT / "data",
    split="test",
    download=(config.execution_mode == ExecutionMode.REAL),
    config=config,
    num_smoke_samples=40,
)
print(f"Loaded {config.execution_mode.value.upper()} images: Train={tuple(train_images.shape)}, Test={tuple(test_images.shape)}")


## 1. Dynamic Entropy Leaf Expansion Ablation

Compare standard MCTS-HNM against MCTS-HNM with dynamic entropy sub-clustering.

In [2]:
# Scenario 1 binary target
target_cls = 0
pos_mask = (train_labels == target_cls)
pos_idx = np.flatnonzero(pos_mask).astype(np.int64)
neg_idx = np.flatnonzero(~pos_mask).astype(np.int64)
bin_labels = pos_mask.astype(np.int64)
test_bin_labels = (test_labels == target_cls).astype(np.int64)

neg_classes = np.unique(train_labels[neg_idx])
class_to_indices = {
    int(c): np.flatnonzero(train_labels == c).astype(np.int64)
    for c in neg_classes
}

tree_static = build_semantic_tree(class_to_indices)
tree_dyn = build_semantic_tree(class_to_indices)

print(f"Initial leaves: {tree_dyn.num_leaves}")
res_static = train_e2e_scenario1(
    train_images=train_images,
    train_binary_labels=bin_labels,
    positive_indices=pos_idx,
    negative_indices=neg_idx,
    test_images=test_images,
    test_binary_labels=test_bin_labels,
    strategy="mcts_enhanced",
    tree=tree_static,
    num_steps=30 if config.execution_mode == ExecutionMode.REAL else 10,
    eval_every=10 if config.execution_mode == ExecutionMode.REAL else 5,
    pos_batch_size=2,
    neg_batch_size=6,
    enable_leaf_expansion=False,
    width_multiplier=0.25,
    device=device,
    seed=42,
)

res_dyn = train_e2e_scenario1(
    train_images=train_images,
    train_binary_labels=bin_labels,
    positive_indices=pos_idx,
    negative_indices=neg_idx,
    test_images=test_images,
    test_binary_labels=test_bin_labels,
    strategy="mcts_enhanced",
    tree=tree_dyn,
    num_steps=30 if config.execution_mode == ExecutionMode.REAL else 10,
    eval_every=10 if config.execution_mode == ExecutionMode.REAL else 5,
    pos_batch_size=2,
    neg_batch_size=6,
    enable_leaf_expansion=True,
    split_threshold=5,
    k_sub=2,
    width_multiplier=0.25,
    device=device,
    seed=42,
)

print(f"Static final F1: {res_static['final_metrics']['f1']:.4f}")
print(f"Dynamic final F1: {res_dyn['final_metrics']['f1']:.4f}, final leaves: {res_dyn['final_num_leaves']}")


Leaf node 5 (class 1001) has only 20 instances (< requested K_mcts=24); sampling with replacement.


Initial leaves: 2
Static final F1: 0.0000
Dynamic final F1: 0.0952, final leaves: 4


/home/mlima/workspace/mcts-hnm/src/training/trainer.py:300: UserWarning: Leaf node 5 (class 1001) has only 20 instances (< requested K_mcts=24); sampling with replacement.
  batch = mcts_sampler.sample_batch(mcts_leaf_id)


## 2. Tier-2 End-to-End ResNet-18 Benchmark

Train ResNet-18 on raw images and evaluate end-to-end performance.

In [3]:
def make_tree():
    return build_semantic_tree(class_to_indices)

e2e_seeds = (42, 43) if config.execution_mode == ExecutionMode.REAL else (42,)
e2e_steps = 100 if config.execution_mode == ExecutionMode.REAL else 10
e2e_eval = 25 if config.execution_mode == ExecutionMode.REAL else 5

cfg = E2EExperimentConfig(
    scenario=1,
    strategies=("uniform", "mcts_enhanced"),
    seeds=e2e_seeds,
    num_steps=e2e_steps,
    eval_every=e2e_eval,
    pos_batch_size=4,
    neg_batch_size=12,
    width_multiplier=0.25,
    output_dir=str(RESULTS_DIR),
    checkpoints_dir=str(CHECKPOINTS_DIR),
)

summary = run_e2e_experiment(
    config=cfg,
    train_images=train_images,
    train_labels=bin_labels,
    test_images=test_images,
    test_labels=test_bin_labels,
    positive_indices=pos_idx,
    negative_indices=neg_idx,
    tree_factory=make_tree,
    device=device,
    experiment_config=config,
)
print("Tier-2 E2E Summary:")
for s, data in summary["strategies"].items():
    print(f"  {s}: mean F1 = {data['mean']:.4f}")


Tier-2 E2E Summary:
  uniform: mean F1 = 0.0000
  mcts_enhanced: mean F1 = 0.0000


## 3. Summary & Verification

Check and verify all exported artifacts in isolated directories.

In [ ]:
print(f"Exported files in {RESULTS_DIR}:")
for f in sorted(RESULTS_DIR.glob("e2e_*")):
    print(f"  {f.name} ({f.stat().st_size} bytes)")
print(f"Exported checkpoints in {CHECKPOINTS_DIR}:")
for f in sorted(CHECKPOINTS_DIR.glob("tier2_*")):
    print(f"  {f.name} ({f.stat().st_size} bytes)")
print("Tier-2 End-to-End vision pipeline execution complete.")
